# risk-management_003

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (174).ipynb`

| Field | Value |
|---|---|
| Section | `risk_management` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 9 |
| Detected functions | calculate_adx, calculate_mfi, process_ticker |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/merged_file.csv')
data

In [ ]:
data = data[data['Ticker'] == "ADANIENT"]
data

In [ ]:
def calculate_adx(high, low, close, period=14):
    tr = np.maximum(high - low, np.maximum(abs(high - close.shift(1)), abs(low - close.shift(1))))
    dm_plus = np.where((high - high.shift(1)) > (low.shift(1) - low), high - high.shift(1), 0)
    dm_minus = np.where((low.shift(1) - low) > (high - high.shift(1)), low.shift(1) - low, 0)
    tr_smooth = pd.Series(tr).rolling(window=period, min_periods=1).mean()
    dm_plus_smooth = pd.Series(dm_plus).rolling(window=period, min_periods=1).mean()
    dm_minus_smooth = pd.Series(dm_minus).rolling(window=period, min_periods=1).mean()
    dx = 100 * abs((dm_plus_smooth - dm_minus_smooth) / (dm_plus_smooth + dm_minus_smooth))
    return dx.rolling(window=period, min_periods=1).mean()

def calculate_mfi(high, low, close, volume, period=14):
    typical_price = (high + low + close) / 3
    money_flow = typical_price * volume
    positive_flow = np.where(typical_price > typical_price.shift(1), money_flow, 0)
    negative_flow = np.where(typical_price < typical_price.shift(1), money_flow, 0)
    positive_mf = pd.Series(positive_flow).rolling(window=period, min_periods=1).sum()
    negative_mf = pd.Series(negative_flow).rolling(window=period, min_periods=1).sum()
    mfi = 100 - (100 / (1 + (positive_mf / negative_mf)))
    return mfi


In [ ]:
# Calculate indicators
data['ADX'] = calculate_adx(data['high'], data['low'], data['close'])
data['MFI'] = calculate_mfi(data['high'], data['low'], data['close'], data['volume'])

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Prepare data
data['date'] = pd.to_datetime(data['date'])
data = data.sort_values(by=['Ticker', 'date'])

# Strategy Parameters
N = 60                         # Accumulation window (30 minutes)
threshold = 0.01              # Volatility threshold (0.5%)
capital = 100000              # Capital per trade (₹1 lakh)
slippage_rate = 0.0001        # 0.01% slippage
brokerage_rate = 0.001        # 0.1% brokerage

tickers = data['Ticker'].unique()

def process_ticker(ticker):
    tradebook = []
    ticker_data = data[data['Ticker'] == ticker].copy()
    ticker_data['day'] = ticker_data['date'].dt.date

    cumulative_pnl = 0
    days = ticker_data['day'].unique()

    for day in days:
        day_data = ticker_data[ticker_data['day'] == day].copy()

        # Calculate rolling indicators
        day_data['std_close'] = day_data['close'].rolling(N).std().shift(1)
        day_data['max_high'] = day_data['high'].rolling(N).max().shift(1)
        day_data['min_low'] = day_data['low'].rolling(N).min().shift(1)
        day_data['avg_volume'] = day_data['volume'].rolling(N).mean().shift(1)
        day_data['avg_close'] = day_data['close'].rolling(N).mean().shift(1)

        trade_cutoff_time = pd.Timestamp(f"{day} 15:15:00")
        exit_time = pd.Timestamp(f"{day} 15:25:00")

        in_trade = False
        trade_type = None
        entry_price = entry_time = shares = stop_loss = take_profit = 0

        for i in range(len(day_data)):
            current_time = day_data['date'].iloc[i]

            # Close trade at end of day
            if in_trade and current_time >= exit_time:
                exit_price = day_data['close'].iloc[i]
                effective_entry = entry_price * (1 + slippage_rate if trade_type == 'BUY' else 1 - slippage_rate)
                effective_exit = exit_price * (1 - slippage_rate if trade_type == 'BUY' else 1 + slippage_rate)
                gross_profit = (effective_exit - effective_entry) * shares * (1 if trade_type == 'BUY' else -1)
                brokerage = brokerage_rate * (entry_price * shares + exit_price * shares)
                net_profit = gross_profit - brokerage
                cumulative_pnl += net_profit

                tradebook.append({
                    'Ticker': ticker, 'Trade Type': trade_type, 'Entry Time': entry_time, 'Entry Price': entry_price,
                    'Exit Time': current_time, 'Exit Price': exit_price, 'Shares': shares,
                    'PnL': net_profit, 'Cumulative PnL': cumulative_pnl, 'Exit Reason': 'End of Day'
                })
                in_trade = False

            elif not in_trade and current_time < trade_cutoff_time:
                if not np.isnan(day_data['std_close'].iloc[i]) and day_data['std_close'].iloc[i] < threshold * day_data['avg_close'].iloc[i] and day_data['volume'].iloc[i] > 1 * day_data['avg_volume'].iloc[i]:
                    if day_data['close'].iloc[i] > day_data['max_high'].iloc[i] and day_data['ADX'].iloc[i] > 25 and day_data['MFI'].iloc[i] > 75:
                        trade_type = 'BUY'
                        entry_price = day_data['close'].iloc[i]
                        shares = int(capital / entry_price)
                        stop_loss = np.min(day_data['low'].iloc[max(0, i - N):i])
                        take_profit = entry_price + 1 * (np.max(day_data['high'].iloc[max(0, i - N):i]) - stop_loss)
                        in_trade = True
                        entry_time = current_time
                    elif day_data['close'].iloc[i] < day_data['min_low'].iloc[i] and day_data['ADX'].iloc[i] > 25 and day_data['MFI'].iloc[i] < 25:
                        trade_type = 'SELL'
                        entry_price = day_data['close'].iloc[i]
                        shares = int(capital / entry_price)
                        stop_loss = np.max(day_data['high'].iloc[max(0, i - N):i])
                        take_profit = entry_price - 1 * (stop_loss - np.min(day_data['low'].iloc[max(0, i - N):i]))
                        in_trade = True
                        entry_time = current_time

    return tradebook

results = Parallel(n_jobs=-1)(delayed(process_ticker)(ticker) for ticker in tqdm(tickers, desc="Processing Tickers"))
tradebook_df = pd.DataFrame([trade for result in results for trade in result])
print(tradebook_df)

# Using existing MFI and ADX from your dataset for faster performance! 🚀 Let me know if you want any adjustments!

In [ ]:
# Optional: Save to CSV
tradebook_df.to_csv('/content/drive/MyDrive/tradebook.csv', index=False)

In [ ]:
import plotly.graph_objects as go

# Plot results with Plotly
fig = go.Figure()
fig.add_trace(go.Candlestick(x=data['date'], open=data['open'], high=data['high'], low=data['low'], close=data['close'], name='Candlestick'))

# Add Buy markers
buy_trades = tradebook_df[tradebook_df['Trade Type'] == 'BUY']
fig.add_trace(go.Scatter(x=buy_trades['Entry Time'], y=buy_trades['Entry Price'], mode='markers', marker=dict(color='green', size=8), name='Buy'))

# Add Sell markers
sell_trades = tradebook_df[tradebook_df['Trade Type'] == 'SELL']
fig.add_trace(go.Scatter(x=sell_trades['Entry Time'], y=sell_trades['Entry Price'], mode='markers', marker=dict(color='red', size=8), name='Sell'))

# Layout settings
fig.update_layout(title='Trading Strategy Visualization', xaxis_title='Time', yaxis_title='Price', xaxis_rangeslider_visible=False)
fig.show()
